# 00 — 三代预训练数据清洗方法论概述

## 第一节：三代方法论演进

| | 第一代（Heuristic） | 第二代（Model-based） | 第三代（Hybrid） |
|---|---|---|---|
| 代表论文 | FineWeb, C4, Gopher (2020-2022) | DCLM (NeurIPS 2024) | Nemotron-CC (NVIDIA 2024) |
| 核心方法 | 人工设计规则 | fastText 分类器 + top-10% | 分类器集成 + Bypass + 改写 |
| 数据保留率 | 30-40% | **~10%** | **~38%** |
| 7B MMLU | ~55% | ~64% (+9%) | ~69% (+5%) |
| 核心优点 | 可解释、极快 | 能评估语义质量 | 高质量+高数量 |
| 核心缺陷 | 无法评估语义质量 | 90%数据丢弃 | API 成本较高 |

方法论演进的核心驱动力：
- 第一代：规则 → 第二代：让模型学会“什么是高质量”
- 第二代：高质量但量少 → 第三代：打破 quality-quantity 的 false trade-off

## 第二节：核心概念术语表（面试必知）

### 1. Quality-Quantity Trade-off
**定义**：过滤越激进，数据质量越高，但数据量越少。两者存在张力。

**DCLM 的发现**：top-10% 过滤使 MMLU 达到 64%，但丢弃了 90% 的数据。

**第三代的突破**：Nemotron-CC 证明这个 trade-off 可以被打破——通过分类器集成和数据改写，在保留 4倍数据的前提下质量不降。

---

### 2. Token Yield（Token 产出率）
**定义**：每处理 1GB 原始数据能产出多少高质量 tokens。

**重要性**：这是比“过滤率”更本质的指标。Nemotron-CC 用 Token Yield 来衡量不同 pipeline 的效率。不同方法的 Token Yield 差异可达 4-10 倍。

**例子**：处理同一批 CC 数据，第二代（top-10%）的 Token Yield = 第三代的 1/4。

---

### 3. Heuristic Filter 的不可组合性
**定义**：10 个 filter 各过滤 5%，总过滤率不是 50%。

**原因一（重叠）**：同一条数据可能被多个 filter 同时命中。

**原因二（顺序依赖）**：先过滤短文档再统计 N-gram，和先统计 N-gram 再过滤短文档，结果不同——后一个会导致 N-gram 统计基于更完整的语料。

**实践意义**：FineWeb 的过滤顺序经过精心设计，不能随意调整。

---

### 4. 代理指标 vs 端到端指标
**代理指标（Proxy Metrics）**：quality 分类器打分、Perplexity、N-gram 多样性——衡量数据本身特征。便宜但间接。

**端到端指标（End-to-End）**：Proxy Model 在 benchmark（MMLU/HellaSwag）的分数——直接衡量数据对模型的影响。可靠但昂贵（需要训练模型）。

**本项目策略**：主要使用代理指标（快速迭代），可选 Proxy Model 验证（Notebook 09）。

---

### 5. Chinchilla Scaling Law
**核心结论**：计算最优的训练配比是 tokens ≈ 参数量 × 20。

**含义**：125M 参数模型需要约 2.5B tokens 才能达到 Chinchilla 最优。7B 模型需要约 140B tokens。

**对数据工程的启示**：第二代（top-10%）的数据量限制了它只能用于小模型/短 horizon 训练。15T token 训练（Llama 3）必须要第三代这样的高 Token Yield 方法。

---

### 6. 循环偏差（Circular Bias）
**定义**：用同一个模型既过滤数据又评估数据质量，产生“自证”的假高分。

**具体案例**：用 fastText(OpenHermes 正样本)过滤后，再用同一个 fastText 打分，分必然很高——但这只说明分类器自洽，不说明数据真的好。

**解决方案**：评估分类器必须与 Pipeline 分类器独立训练（不同正样本、不同超参）。

---

### 7. Distribution Shift（分布偏移）
**定义**：过滤不仅改变数据量，还改变数据分布。

**典型案例**：激进质量过滤 → 百科/学术内容富集 → MMLU 分高，但对话/创作能力下降。

**量化方法**：用 N-gram unique ratio 和域名 Shannon Entropy 跟踪分布变化。

---

### 8. Data Mixing（数据配比）
**定义**：清洗后将不同来源的数据按比例混合（Web + Code + Books + Wiki）。

**Llama 3 的配比（参考）**：Web 50% + Code 25% + 学术 15% + Wikipedia 5% + 其他 5%。

**本项目定位**：我们聚焦“清洗过滤”这一步，Data Mixing 是下游环节。

## 第三节：本项目实验设计

### 数据来源
| 数据集 | 用途 | 规模 |
|---|---|---|
| Common Crawl WARC | 主实验数据（三代 pipeline 共用） | smoke_test: 1K 文档 / full_run: 5W 文档 |
| Wikipedia 摘要 | 正样本（分类器训练 + 评估） | 1万条 |
| StackExchange ELI5 | 正样本（Gen2 分类器） | 5千条 |
| FineWeb sample-10BT | 对比参考（已经过 FineWeb 处理） | 1 parquet 分片 |

### 评估方法
1. **代理指标**（主要）：质量分类器打分、Perplexity、N-gram 多样性、毒性
2. **可选端到端验证**（Notebook 09）：GPT-2 125M Proxy Model + benchmark

### 预期假设（基于论文）
- 第二代 quality_score > 第一代 > 原始数据
- 第三代 quality_score ≈ 第二代，但数据保留率是第二代的 3-4 倍
- Heuristic bypass 的误杀率约为 15-20%（对标 Nemotron-CC 的 18.1%）

In [1]:
# 导入配置并打印当前运行模式
import sys
sys.path.insert(0, '..')
from src.utils.config_loader import load_run_config, print_config_summary

run_cfg = load_run_config()
print_config_summary(run_cfg)

  当前运行模式: SMOKE_TEST
  10-15分钟跑完全流程，验证代码无报错
──────────────────────────────────────────────────
  doc_limit       : 1,000
  eval_sample_size: 200
  audit_sample_size: 20
  rewrite_count   : 50
  random_seed     : 42
